# PDF Parsing with Infinity-Parser2-Pro

Parses every PDF under `data/textbook/` with [infly/Infinity-Parser2-Pro](https://huggingface.co/infly/Infinity-Parser2-Pro) — a 35B-param Qwen3.5-based VLM for document parsing (layout + text + tables in one pass, output as Markdown or JSON w/ bboxes).

**Needs a CUDA GPU or an Apple Silicon Mac with enough unified memory.** 35B params in BF16 is ~70GB of weights alone.
- CUDA box: A100/H100-class, enough VRAM for 35B BF16 (or 4-bit/8-bit via `bitsandbytes` if VRAM-constrained).
- Apple Silicon (e.g. Mac Studio): 128GB+ unified memory recommended — runs via PyTorch's MPS backend (`device="mps"`). No `flash_attention_2` on MPS (CUDA-only kernel), so attention runs at eager/sdpa speed — slower than a CUDA box, but functional.

Does **not** run on memory-constrained machines (e.g. 24GB M-series laptops) — the ~70GB weights alone won't fit.

Recommended deps (not in this project's `pyproject.toml` — install separately, e.g. into a `.venv-infinity-parser` venv):
```
uv venv .venv-infinity-parser --python 3.13
# CUDA box:
uv pip install --python .venv-infinity-parser \
    torch --index-url https://download.pytorch.org/whl/cu124 \
    transformers accelerate qwen-vl-utils pillow pymupdf tqdm
# Apple Silicon (Mac Studio etc.):
uv pip install --python .venv-infinity-parser \
    torch transformers accelerate qwen-vl-utils pillow pymupdf tqdm
# optional (CUDA only), for lower VRAM: bitsandbytes
# optional (CUDA only), for max throughput: flash-attn, vllm
```

Unlike the MinerU+Qwen-VL dual pipeline (`dual_pipeline_parsing.ipynb`), Infinity-Parser2-Pro is a single model producing full-page Markdown directly — output here is saved standalone under `output/<pdf_stem>/infinity_parser/`, not merged into the MinerU-shaped `post_process` pipeline (different schema).

## Setup

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/

TEXTBOOK_DIR = PROJECT_ROOT / "data" / "textbook"
OUTPUT_ROOT = PROJECT_ROOT / "output"

pdf_paths = sorted(TEXTBOOK_DIR.rglob("*.pdf"))
assert pdf_paths, f"No PDFs found under {TEXTBOOK_DIR}"

len(pdf_paths), pdf_paths[0]

## Render PDF pages to images

Rasterizes each PDF page to PNG via PyMuPDF, at 150 DPI.

In [ ]:
import fitz  # PyMuPDF

RENDER_DPI = 150


def render_pages(pdf_path: Path, output_dir: Path, dpi: int = RENDER_DPI) -> list[Path]:
    """Renders every page of `pdf_path` to a PNG in `output_dir`, named
    `page_{page_idx:04d}.png` (0-indexed). Returns image paths in page order."""
    output_dir.mkdir(parents=True, exist_ok=True)
    zoom = dpi / 72  # PDF points are 72 per inch

    paths = []
    with fitz.open(pdf_path) as doc:
        for page_idx, page in enumerate(doc):
            pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
            image_path = output_dir / f"page_{page_idx:04d}.png"
            pix.save(image_path)
            paths.append(image_path)

    return paths

## Load Infinity-Parser2-Pro

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # Apple Silicon, e.g. Mac Studio unified memory
else:
    raise RuntimeError(
        "No CUDA or MPS device found. Infinity-Parser2-Pro (35B, ~70GB bf16) "
        "needs a CUDA GPU or an Apple Silicon Mac with enough unified memory "
        "(128GB+ recommended)."
    )

MODEL_PATH = "infly/Infinity-Parser2-Pro"

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    # attn_implementation="flash_attention_2",  # CUDA only, not available on MPS
)
model.to(DEVICE)
model.eval()

## Parse function

Uses Infinity-Parser2-Pro's JSON layout mode (`doc2json`-style prompt) instead of direct Markdown, so each layout element comes back with a `category`, `text`, and `bbox` — needed to crop out `figure`/`table` elements as separate image files (see "Crop visuals" below).

`bbox` scale is not documented on the model card (pixel vs normalized, and relative to which image). To stay correct regardless, we read back the **actual resized image size the processor fed the model** (`image_inputs[0].size`, post `qwen_vl_utils` smart-resize) and scale bboxes from that space into the original page-render pixel space before cropping — rather than assuming a fixed convention.

In [ ]:
import json
import re

from PIL import Image
from qwen_vl_utils import process_vision_info

PARSE_PROMPT = (
    "Extract layout information from this document page image. For each "
    "layout element, output its bbox, category, and the text content "
    "within the bbox. Bbox format: [x1, y1, x2, y2] in pixel coordinates "
    "of this image. Categories: header, title, text, figure, table, "
    "formula, figure_caption, table_caption, formula_caption, "
    "figure_footnote, table_footnote, page_footnote, footer. "
    "Return a JSON array of objects with keys \"category\", \"bbox\", "
    "\"text\". For figure/table elements, leave \"text\" empty and instead "
    "put a short description of the visual in a \"caption\" key. Output "
    "only the JSON array, no other text."
)

_JSON_ARRAY_RE = re.compile(r"\[.*\]", re.DOTALL)


def parse_page(image_path: Path, max_new_tokens: int = 4096) -> tuple[list[dict], tuple[int, int]]:
    """Returns (elements, model_image_size). model_image_size is the
    (width, height) of the image actually fed to the model after
    qwen_vl_utils' smart-resize — bboxes in `elements` are in that space."""
    page_image = Image.open(image_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": PARSE_PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None
        )

    generated_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw = processor.batch_decode(
        generated_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    match = _JSON_ARRAY_RE.search(raw)
    if not match:
        raise ValueError(f"No JSON array found in model output for {image_path}:\n{raw[:500]}")
    elements = json.loads(match.group(0))

    model_image_size = image_inputs[0].size  # (width, height), PIL convention
    return elements, model_image_size

## Crop visuals into separate files

Crops `figure`/`table` elements out of the full-resolution page render (not the possibly-downscaled model input) and saves each as its own PNG under `<out_dir>/images/`. Also builds the page's Markdown by walking elements in order, replacing each `figure`/`table` bbox with an image reference + caption instead of transcribed text.

In [ ]:
VISUAL_CATEGORIES = {"figure", "table"}


def crop_visuals_and_render_markdown(
    elements: list[dict],
    model_image_size: tuple[int, int],
    page_image_path: Path,
    images_out_dir: Path,
    page_idx: int,
) -> str:
    """Crops figure/table elements from the full-res page render (scaling
    bboxes from model_image_size into the render's actual pixel size) and
    saves them as `page_{page_idx:04d}_elem_{i:02d}.png` under
    images_out_dir. Returns page Markdown with visuals replaced by image
    references."""
    page_image = Image.open(page_image_path).convert("RGB")
    render_w, render_h = page_image.size
    model_w, model_h = model_image_size
    scale_x = render_w / model_w
    scale_y = render_h / model_h

    images_out_dir.mkdir(parents=True, exist_ok=True)

    lines = []
    for i, elem in enumerate(elements):
        category = elem.get("category", "text")
        bbox = elem.get("bbox")

        if category in VISUAL_CATEGORIES and bbox and len(bbox) == 4:
            x1, y1, x2, y2 = bbox
            crop_box = (
                max(0, round(x1 * scale_x)),
                max(0, round(y1 * scale_y)),
                min(render_w, round(x2 * scale_x)),
                min(render_h, round(y2 * scale_y)),
            )
            if crop_box[2] <= crop_box[0] or crop_box[3] <= crop_box[1]:
                continue  # degenerate bbox, skip

            crop = page_image.crop(crop_box)
            crop_filename = f"page_{page_idx:04d}_elem_{i:02d}.png"
            crop.save(images_out_dir / crop_filename)

            caption = elem.get("caption", "").strip()
            alt = caption or category
            lines.append(f"![{alt}](images/{crop_filename})")
            if caption:
                lines.append(f"*{caption}*")
        else:
            text = (elem.get("text") or "").strip()
            if text:
                lines.append(text)

    return "\n\n".join(lines)

## Smoke test on one PDF, one page

In [ ]:
sample_pdf = pdf_paths[0]
sample_out_dir = OUTPUT_ROOT / sample_pdf.stem / "infinity_parser"
sample_pages = render_pages(sample_pdf, sample_out_dir / "rendered_pages")

sample_elements, sample_model_size = parse_page(sample_pages[0])
sample_md = crop_visuals_and_render_markdown(
    sample_elements, sample_model_size, sample_pages[0], sample_out_dir / "images", page_idx=0
)
print(sample_md)

## Batch: parse every PDF under `data/textbook/`

For each PDF: render pages, parse each page to layout elements, crop `figure`/`table` elements to `output/<pdf_stem>/infinity_parser/images/`, concatenate per-page Markdown (with image refs) into one `.md` file. Skips PDFs whose output already exists, so this cell is safe to re-run after an interruption.

In [ ]:
from tqdm.auto import tqdm

PAGE_BREAK = "\n\n---\n\n"

for pdf_path in tqdm(pdf_paths, desc="PDFs"):
    out_dir = OUTPUT_ROOT / pdf_path.stem / "infinity_parser"
    out_md_path = out_dir / f"{pdf_path.stem}.md"
    if out_md_path.exists():
        continue

    page_image_paths = render_pages(pdf_path, out_dir / "rendered_pages")
    images_out_dir = out_dir / "images"

    page_markdowns = []
    for page_idx, image_path in enumerate(tqdm(page_image_paths, desc=pdf_path.stem, leave=False)):
        elements, model_image_size = parse_page(image_path)
        page_md = crop_visuals_and_render_markdown(
            elements, model_image_size, image_path, images_out_dir, page_idx=page_idx
        )
        page_markdowns.append(page_md)

    out_md_path.write_text(PAGE_BREAK.join(page_markdowns), encoding="utf-8")